# How Tool Calling Works

Tool calling is a **request–response loop** between your application and a language model:

1. You send **messages** (system, user, prior turns) plus **tool schemas**.
2. The model returns either **natural language** or a structured **tool call** (`name` + `arguments`).
3. **Your runtime** executes the tool — the model never runs code itself.
4. You append the result as a **tool message** (observation) and call the model again.
5. Repeat until the model returns a final answer or you hit `max_steps`.

This notebook walks through every piece of that loop: message shapes, execution, re-prompting, parallel tool calls, trace validation, and guardrails. You will build a **manual tool loop** from scratch — no agent framework required — using a fictional **Order Support API** with real tool functions.

Everything is self-contained below. No prior notebooks or external services are required.

In [ ]:
"""
Order Support API — environment for tool-calling demos.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- In-memory backend (simulates Order Support API) ---
PRODUCT_CATALOG: list[dict[str, Any]] = [
    {"sku": "LAP-001", "name": "Pro Laptop 14", "price": 1299.00, "stock": 12},
    {"sku": "MON-002", "name": "4K Monitor 27", "price": 449.00, "stock": 34},
    {"sku": "KEY-003", "name": "Mechanical Keyboard", "price": 129.00, "stock": 0},
]

ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {
        "customer": "alice@shop.com",
        "items": [{"sku": "LAP-001", "qty": 1}],
        "status": "shipped",
        "tracking": "1Z999AA10123456784",
    },
}

POLICY_DOCS: list[dict[str, str]] = [
    {
        "id": "ship-01",
        "text": "Standard shipping takes 5-7 business days. Express shipping (2 days) costs $15 extra.",
    },
    {
        "id": "ret-02",
        "text": "Returns accepted within 30 days if item is unused. Refunds process in 5 business days.",
    },
    {
        "id": "stock-03",
        "text": "Out-of-stock items can be backordered. Estimated restock date shown on product page.",
    },
]

print(f"Products: {len(PRODUCT_CATALOG)} | Orders: {len(ORDERS)} | Policy docs: {len(POLICY_DOCS)}")

---

## 1. The Request–Response Tool Loop

```
messages ──► model (+ tool schemas) ──► tool_calls?
                                            │ yes
                                            ▼
                                      execute tools locally
                                            │
                                            ▼
                               append tool message(s) to messages
                                            │
                                            └──► model again ──► final text or more tools
```

The loop is the same whether you use the raw OpenAI API, Anthropic, or any agent framework. Frameworks wrap this loop; they do not change the underlying contract:

- The **model proposes** actions.
- The **runtime disposes** (validates, executes, logs).
- **Observations** flow back as tool messages.

---

## 2. Implementing the Tools

Before wiring the loop, define the actual Python functions your agent can call. Each tool should have clear inputs, structured outputs, and validation at the boundary.

In [ ]:
def search_products(query: str) -> list[dict[str, Any]]:
    """Search product catalog by name or SKU keyword."""
    q = query.lower()
    return [
        p for p in PRODUCT_CATALOG
        if q in p["name"].lower() or q in p["sku"].lower()
    ]


def lookup_order(order_id: str) -> dict[str, Any]:
    """Return order details or not-found error."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return {"found": False, "error": f"Order {order_id} not found"}
    return {"found": True, "order_id": order_id.upper(), **order}


def search_policy(query: str) -> list[dict[str, str]]:
    """Search shipping/returns policy documents."""
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    hits = []
    for doc in POLICY_DOCS:
        if any(t in doc["text"].lower() for t in terms):
            hits.append(doc)
    return hits or POLICY_DOCS[:1]


def create_support_ticket(customer_email: str, subject: str, body: str) -> dict[str, Any]:
    """Create a support ticket (simulated)."""
    if "@" not in customer_email:
        return {"success": False, "error": "Invalid email address"}
    ticket_id = f"TKT-{uuid.uuid4().hex[:6].upper()}"
    return {
        "success": True,
        "ticket_id": ticket_id,
        "customer": customer_email,
        "subject": subject,
        "body": body,
        "created_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "search_products": search_products,
    "lookup_order": lookup_order,
    "search_policy": search_policy,
    "create_support_ticket": create_support_ticket,
}

# Quick sanity check
print("search_products('laptop'):", search_products("laptop"))
print("lookup_order('ORD-1001'):", lookup_order("ORD-1001")["status"])

---

## 3. Tool Schemas — What the Model Sees

Tool schemas are JSON descriptions sent to the API. They tell the model what tools exist, when to use them, and what arguments they accept. The model uses schemas to generate structured `tool_calls` — not free-form text.

In [ ]:
TOOL_SCHEMAS: list[dict[str, Any]] = [
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search the product catalog by name or SKU.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string", "description": "Product name or SKU keyword."}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": "Look up an existing order by order ID (e.g. ORD-1001).",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string"}},
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_policy",
            "description": "Search shipping, returns, and stock policy documents.",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_support_ticket",
            "description": "Create a customer support ticket when the issue needs human follow-up.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_email": {"type": "string"},
                    "subject": {"type": "string"},
                    "body": {"type": "string"},
                },
                "required": ["customer_email", "subject", "body"],
            },
        },
    },
]

print(f"Registered {len(TOOL_REGISTRY)} tools with {len(TOOL_SCHEMAS)} schemas")

---

## 4. Message Types in the Loop

| Message | Role | Carries |
|---------|------|--------|
| System | `system` | Agent behavior rules |
| User | `user` | Customer request |
| Assistant (tool turn) | `assistant` | `tool_calls` list; `content` is often `null` |
| Tool result | `tool` | `tool_call_id` + observation string |
| Assistant (final) | `assistant` | Natural-language answer to the user |

The **full transcript** — every message in order — is what you send on each model call. Providers bill against this growing context.

In [ ]:
messages: list[dict[str, Any]] = [
    {
        "role": "system",
        "content": (
            "You are an order support assistant. Use tools to look up orders, "
            "search products and policies. Be concise and cite policy doc IDs."
        ),
    },
    {
        "role": "user",
        "content": "Where is my order ORD-1001 and what is your return policy?",
    },
]

print(f"Initial transcript: {len(messages)} messages")
for i, m in enumerate(messages):
    print(f"  [{i}] {m['role']}: {m['content'][:60]}...")

---

## 5. Assistant `tool_calls` Structure

When the model decides to use a tool, it returns an assistant message with a `tool_calls` array. Each entry has:

- **`id`** — unique identifier (you must echo this in the tool response)
- **`type`** — always `"function"` for OpenAI-compatible APIs
- **`function.name`** — which tool to invoke
- **`function.arguments`** — JSON string of parameters

In [ ]:
assistant_tool_turn = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": "call_ord_lookup",
            "type": "function",
            "function": {
                "name": "lookup_order",
                "arguments": json.dumps({"order_id": "ORD-1001"}),
            },
        },
        {
            "id": "call_policy_search",
            "type": "function",
            "function": {
                "name": "search_policy",
                "arguments": json.dumps({"query": "returns refund"}),
            },
        },
    ],
}

print("Model emitted parallel tool calls:")
print(pretty(assistant_tool_turn))

---

## 6. Tool Messages — Returning Observations

After executing each tool call, append one **tool message** per call:

- **`role`**: `"tool"`
- **`tool_call_id`**: must match the `id` from the assistant's `tool_calls`
- **`content`**: string result (usually JSON) the model reads on the next turn

Observations should be **compact and truthful**. Do not summarize or embellish — the model will reason over exactly what you return.

In [ ]:
def execute_tool_call(tool_call: dict[str, Any]) -> dict[str, Any]:
    """Dispatch one tool call to the registry and format a tool message."""
    fn_info = tool_call["function"]
    name = fn_info["name"]
    args = json.loads(fn_info["arguments"])

    if name not in TOOL_REGISTRY:
        content = json.dumps({"error": f"Unknown tool: {name}"})
    else:
        try:
            result = TOOL_REGISTRY[name](**args)
            content = result if isinstance(result, str) else json.dumps(result, default=str)
        except Exception as exc:
            content = json.dumps({"error": str(exc)})

    return {
        "role": "tool",
        "tool_call_id": tool_call["id"],
        "content": content,
    }


# Execute both parallel calls
tool_messages = [execute_tool_call(tc) for tc in assistant_tool_turn["tool_calls"]]
print(f"Generated {len(tool_messages)} tool messages for 1 assistant turn:\n")
for tm in tool_messages:
    print(f"  id={tm['tool_call_id']}: {tm['content'][:80]}...")

---

## 7. Execute and Re-Prompt

After appending the assistant message and all tool messages, call the model **again** with the **full** transcript. The model typically either:

- Returns a **final natural-language answer**, or
- Emits **more tool calls** if it needs additional information.

Below we walk through one complete turn manually, then automate the full loop.

In [ ]:
# Build transcript: system + user + assistant(tool_calls) + tool results + final answer
transcript: list[dict[str, Any]] = list(messages)
transcript.append(assistant_tool_turn)
transcript.extend(tool_messages)

assistant_final = {
    "role": "assistant",
    "content": (
        "Order ORD-1001 is shipped (tracking 1Z999AA10123456784). "
        "Returns are accepted within 30 days if unused [ret-02]; refunds process in 5 business days."
    ),
}
transcript.append(assistant_final)

print("Full transcript after one tool round:\n")
for i, m in enumerate(transcript):
    role = m["role"]
    if m.get("tool_calls"):
        names = [tc["function"]["name"] for tc in m["tool_calls"]]
        preview = f"tool_calls={names}"
    else:
        preview = (m.get("content") or "")[:70]
    print(f"  [{i}] {role:10} {preview}")

---

## 8. Manual Tool Loop — Full Implementation

This is the core pattern every agent framework implements. You control iteration limits, logging, and which model client to use.

`OfflineModel` simulates what a real LLM would return based on conversation state — swap it for `client.chat.completions.create(..., tools=TOOL_SCHEMAS)` in production.

In [ ]:
class OfflineModel:
    """Rule-based stand-in for an LLM that emits tool_calls then a final answer."""

    def __call__(self, messages: list[dict[str, Any]]) -> dict[str, Any]:
        tool_results = [m for m in messages if m.get("role") == "tool"]
        user_msg = next(m["content"] for m in messages if m["role"] == "user")

        if not tool_results:
            # First turn: parallel lookup based on user intent
            calls = []
            if "ord" in user_msg.lower() or "order" in user_msg.lower():
                match = re.search(r"ORD-\d+", user_msg.upper())
                order_id = match.group(0) if match else "ORD-1001"
                calls.append({
                    "id": "call_lookup",
                    "type": "function",
                    "function": {
                        "name": "lookup_order",
                        "arguments": json.dumps({"order_id": order_id}),
                    },
                })
            if "return" in user_msg.lower() or "policy" in user_msg.lower():
                calls.append({
                    "id": "call_policy",
                    "type": "function",
                    "function": {
                        "name": "search_policy",
                        "arguments": json.dumps({"query": "returns"}),
                    },
                })
            if "keyboard" in user_msg.lower() or "stock" in user_msg.lower():
                calls.append({
                    "id": "call_product",
                    "type": "function",
                    "function": {
                        "name": "search_products",
                        "arguments": json.dumps({"query": "keyboard"}),
                    },
                })
            if not calls:
                calls.append({
                    "id": "call_default",
                    "type": "function",
                    "function": {
                        "name": "search_policy",
                        "arguments": json.dumps({"query": "shipping"}),
                    },
                })
            return {"role": "assistant", "content": None, "tool_calls": calls}

        # After tools: check if we need a ticket for out-of-stock
        if len(tool_results) == 1 and any('"stock": 0' in tr["content"] for tr in tool_results):
            return {
                "role": "assistant",
                "content": None,
                "tool_calls": [{
                    "id": "call_ticket",
                    "type": "function",
                    "function": {
                        "name": "create_support_ticket",
                        "arguments": json.dumps({
                            "customer_email": "customer@shop.com",
                            "subject": "Backorder request: Mechanical Keyboard",
                            "body": "Customer asked about out-of-stock KEY-003. Request restock notification.",
                        }),
                    },
                }],
            }

        # Final answer
        snippets = [tr["content"][:100] for tr in tool_results]
        return {
            "role": "assistant",
            "content": f"Here is what I found: {' | '.join(snippets)}",
        }


def run_tool_loop(
    user_query: str,
    model: OfflineModel | None = None,
    max_steps: int = 6,
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Manual agent loop — the pattern all frameworks wrap."""
    model = model or OfflineModel()
    msgs: list[dict[str, Any]] = [
        {"role": "system", "content": "You are an order support assistant. Use tools when needed."},
        {"role": "user", "content": user_query},
    ]
    trace: list[dict[str, Any]] = []

    for step in range(max_steps):
        ai_turn = model(msgs)
        msgs.append(ai_turn)
        trace.append({
            "step": step,
            "type": "assistant",
            "tool_calls": [tc["function"]["name"] for tc in ai_turn.get("tool_calls", [])],
            "content": ai_turn.get("content"),
        })

        if not ai_turn.get("tool_calls"):
            break  # final answer — stop loop

        for tc in ai_turn["tool_calls"]:
            tool_msg = execute_tool_call(tc)
            msgs.append(tool_msg)
            trace.append({
                "step": step,
                "type": "tool",
                "name": tc["function"]["name"],
                "tool_call_id": tc["id"],
                "content_preview": tool_msg["content"][:80],
            })

    return msgs, trace


final_msgs, trace = run_tool_loop("Where is order ORD-1001 and what is the return policy?")
print(f"Loop completed in {len(trace)} trace events")
print(f"Final answer: {final_msgs[-1]['content']}")

---

## 9. Multi-Step Trace — Observability

Production agents log every model and tool turn. The trace below is what you would send to an observability platform.

In [ ]:
def print_trace(trace: list[dict[str, Any]]) -> None:
    print(f"{'Step':>4} {'Type':<12} {'Detail'}")
    print("-" * 70)
    for event in trace:
        step = event.get("step", "-")
        etype = event["type"]
        if etype == "assistant":
            if event.get("tool_calls"):
                detail = f"tool_calls={event['tool_calls']}"
            else:
                detail = f"answer={str(event.get('content', ''))[:50]}"
        else:
            detail = f"{event.get('name')} → {event.get('content_preview', '')}"
        print(f"{step:>4} {etype:<12} {detail}")


print_trace(trace)

---

## 10. Multi-Step Example — Out-of-Stock Triggers a Ticket

A realistic loop often needs **more than one round**: search product → see zero stock → create support ticket → summarize for the customer.

In [ ]:
msgs2, trace2 = run_tool_loop("Is the mechanical keyboard in stock? If not, open a support ticket.")
print_trace(trace2)
print(f"\nFinal: {msgs2[-1]['content']}")

---

## 11. Parallel Tool Calls in One Turn

Models often emit **multiple** `tool_calls` in a single assistant message. Rules:

1. Execute **all** tool calls (they are independent unless your domain says otherwise).
2. Append **one tool message per call**, each with the matching `tool_call_id`.
3. Re-prompt the model **once** with all observations — not once per tool.

In [ ]:
parallel_turn = {
    "role": "assistant",
    "content": None,
    "tool_calls": [
        {
            "id": "call_a",
            "type": "function",
            "function": {"name": "lookup_order", "arguments": json.dumps({"order_id": "ORD-1001"})},
        },
        {
            "id": "call_b",
            "type": "function",
            "function": {"name": "search_policy", "arguments": json.dumps({"query": "shipping"})},
        },
        {
            "id": "call_c",
            "type": "function",
            "function": {"name": "search_products", "arguments": json.dumps({"query": "monitor"})},
        },
    ],
}

parallel_results = [execute_tool_call(tc) for tc in parallel_turn["tool_calls"]]
print(f"{len(parallel_turn['tool_calls'])} tool calls → {len(parallel_results)} tool messages")
for tm in parallel_results:
    print(f"  [{tm['tool_call_id']}] {tm['content'][:60]}...")

---

## 12. Transcript Validation

A common bug is **orphan tool messages** — a tool result whose `tool_call_id` does not match any pending call. Validate before sending to the API.

In [ ]:
def validate_transcript(messages: list[dict[str, Any]]) -> tuple[bool, str]:
    """Ensure every tool message pairs with a tool_call id."""
    pending: set[str] = set()
    for m in messages:
        if m.get("tool_calls"):
            for tc in m["tool_calls"]:
                pending.add(tc["id"])
        if m.get("role") == "tool":
            tid = m.get("tool_call_id")
            if tid not in pending:
                return False, f"Orphan tool message: {tid} not in pending {pending}"
            pending.discard(tid)
    if pending:
        return False, f"Unanswered tool calls: {pending}"
    return True, "valid"


ok, msg = validate_transcript(final_msgs)
print(f"Order lookup transcript: {msg}")

ok2, msg2 = validate_transcript(transcript)
print(f"Manual build transcript: {msg2}")

# Broken example
broken = [
    {"role": "assistant", "tool_calls": [{"id": "call_x", "type": "function", "function": {"name": "x", "arguments": "{}"}}]},
    {"role": "tool", "tool_call_id": "call_WRONG", "content": "oops"},
]
ok3, msg3 = validate_transcript(broken)
print(f"Broken transcript: {msg3}")

---

## 13. Guardrails — Duplicate Detection and Step Limits

Agents can get stuck calling the same tool with the same arguments repeatedly. Track call signatures and stop when duplicates exceed a threshold.

In [ ]:
@dataclass
class ToolLoopGuardrails:
    max_steps: int = 6
    max_duplicate_calls: int = 2
    _call_counts: dict[str, int] = field(default_factory=dict)

    def record_call(self, name: str, arguments: str) -> bool:
        """Return False if duplicate limit exceeded."""
        key = f"{name}:{arguments}"
        self._call_counts[key] = self._call_counts.get(key, 0) + 1
        return self._call_counts[key] <= self.max_duplicate_calls


def run_tool_loop_safe(user_query: str, max_steps: int = 6) -> dict[str, Any]:
    guard = ToolLoopGuardrails(max_steps=max_steps)
    model = OfflineModel()
    msgs: list[dict[str, Any]] = [
        {"role": "system", "content": "Order support assistant."},
        {"role": "user", "content": user_query},
    ]

    for step in range(guard.max_steps):
        ai_turn = model(msgs)
        msgs.append(ai_turn)
        if not ai_turn.get("tool_calls"):
            return {"status": "done", "answer": ai_turn["content"], "steps": step + 1}

        for tc in ai_turn["tool_calls"]:
            args = tc["function"]["arguments"]
            if not guard.record_call(tc["function"]["name"], args):
                return {"status": "blocked", "reason": "duplicate tool call limit", "steps": step + 1}
            msgs.append(execute_tool_call(tc))

    return {"status": "max_steps", "reason": f"exceeded {max_steps} steps", "steps": max_steps}


result = run_tool_loop_safe("Where is order ORD-1001?")
print(result)

---

## 14. Native Tool API vs ReAct Text Pattern

**ReAct** interleaves reasoning (`Thought:`) and actions (`Action:`) as **plain text**. The runtime parses that text to find tool names and arguments.

**Native tool calling** uses structured `tool_calls` JSON returned by the API — more reliable when the provider supports it.

| | Native tool API | ReAct (text) |
|--|-----------------|--------------|
| Action format | `tool_calls` JSON | `Action: search_products[...]` |
| Thought visibility | Often internal to model | Explicit `Thought:` lines |
| Reliability | High with good schemas | Parser fragility |
| Best for | Production agents | Teaching, legacy models |

The manual loop you built is the **native tool API** pattern. The state machine is identical; only the parsing layer differs.

In [ ]:
REACT_TRACE = [
    "Thought: Customer asked about order ORD-1001 and returns.",
    'Action: lookup_order({"order_id": "ORD-1001"})',
    "Observation: status=shipped, tracking=1Z999AA10123456784",
    'Action: search_policy({"query": "returns"})',
    "Observation: [ret-02] Returns accepted within 30 days...",
    "Thought: I have enough to answer.",
    "Final Answer: Your order is shipped. Returns allowed within 30 days.",
]

print("ReAct (text parsing required):")
for line in REACT_TRACE:
    print(f"  {line}")

print(f"\nNative tool API trace: {len(trace)} structured events (no regex parsing)")

---

## 15. Observation Formatting Best Practices

| Practice | Why |
|----------|-----|
| Return JSON strings for structured data | Model parses fields reliably |
| Include error keys on failure | Model can retry or explain to user |
| Truncate large payloads | Protect context window |
| Never fabricate results in the executor | Model trusts observations as ground truth |
| Keep field names stable across versions | Avoid breaking model reasoning |

In [ ]:
def format_observation(result: Any, max_chars: int = 500) -> str:
    """Standardize tool output before appending as tool message."""
    if isinstance(result, str):
        text = result
    else:
        text = json.dumps(result, default=str)
    if len(text) > max_chars:
        return text[:max_chars] + f"... [truncated, {len(text)} chars total]"
    return text


big_catalog = {"products": PRODUCT_CATALOG * 50}
print(format_observation(big_catalog, max_chars=200))

---

## 16. Optional — Live OpenAI Tool Loop

Set `USE_LIVE_LLM = True` with a valid API key. The control flow is identical to `run_tool_loop` — only the `model` function changes.

In [ ]:
USE_LIVE_LLM = False


def live_openai_tool_loop(user_query: str, max_steps: int = 5) -> str:
    """Reference implementation using OpenAI chat completions API."""
    from openai import OpenAI

    client = OpenAI()
    msgs: list[dict[str, Any]] = [
        {"role": "system", "content": "You are an order support assistant. Use tools."},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=msgs,
            tools=TOOL_SCHEMAS,
        )
        msg = response.choices[0].message
        assistant_entry: dict[str, Any] = {"role": "assistant", "content": msg.content}
        if msg.tool_calls:
            assistant_entry["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                }
                for tc in msg.tool_calls
            ]
        msgs.append(assistant_entry)

        if not msg.tool_calls:
            return msg.content or ""

        for tc in msg.tool_calls:
            tool_call_dict = {
                "id": tc.id,
                "function": {"name": tc.function.name, "arguments": tc.function.arguments},
            }
            msgs.append(execute_tool_call(tool_call_dict))

    return "Stopped: max_steps reached"


if USE_LIVE_LLM:
    try:
        answer = live_openai_tool_loop("Where is order ORD-1001?")
        print(answer)
    except Exception as exc:
        print("LLM call failed:", exc)
else:
    print(
        "Offline mode: run_tool_loop() above demonstrates identical control flow. "
        "Set USE_LIVE_LLM=True to call OpenAI with the same message discipline."
    )

---

## 17. Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Orphan tool message | API error, wrong pairing | Match `tool_call_id` exactly |
| Re-prompt before all tools return | Model sees incomplete state | Append all tool messages first |
| Infinite tool loop | Cost spike | `max_steps` + duplicate detection |
| Huge tool payloads | Context overflow | Truncate/summarize observations |
| Model executes tools | Security risk | Only your runtime calls `TOOL_REGISTRY` |
| Skipping validation | Bad args reach production | Validate at tool boundary |

---

## 18. Full Transcript Dump

Inspect the complete message list after the loop — this is what providers bill against and what you persist for audit.

In [ ]:
for i, m in enumerate(final_msgs):
    print(f"--- message {i} ({m['role']}) ---")
    print(pretty(m))
    print()

---

## 19. Check Your Understanding

1. Who executes a tool call — the model or your runtime?
2. What three fields does each entry in `tool_calls` need?
3. Why must `tool_call_id` in a tool message match the assistant's `id`?
4. When a model emits three parallel tool calls, how many tool messages do you append before re-prompting?
5. What is the difference between native tool calling and ReAct text parsing?
6. Name two guardrails that prevent runaway tool loops.
7. Why should tool observations be compact and never fabricated?

---

## 20. Summary

Tool calling is a **loop**: assistant proposes `tool_calls` → runtime executes → tool messages return observations → model is called again until it produces a final answer.

**Key takeaways:**

- The **model proposes**; the **runtime executes**. Never let the model call APIs directly.
- Each tool call needs a matching **tool message** with the same `tool_call_id`.
- **Parallel tool calls** in one turn require all results before the next model call.
- A **manual loop** with `max_steps`, duplicate detection, and transcript validation is the foundation every framework builds on.
- **Native tool APIs** are more reliable than parsing ReAct text for production.
- **Observations** must be truthful, structured, and compact — the model treats them as ground truth.

You now understand the full request–response cycle. The next step in your learning is designing tool schemas the model can invoke reliably.